# Reconstruct Ali Mansour Profile Dates & Create Temporal Train/Test Split

## Objective

The team lead's final profiling dataset contains profiles from all 15 days,
but the original `profile_date` column is missing.

We will reconstruct the profile date by matching:

    Team lead: calling_number
        ↓
    My CDR data: msisdn + answer_date

This allows us to reconstruct:

- Days 1–12 → training set
- Days 13–15 → test set

The reconstructed split will then be used to train and evaluate the same
SIMBox fraud detection model on the team lead's dataset.

## Important

The team lead's dataset has:
- profiles with <=10 outgoing calls already removed
- fraud labels partially relabeled as normal
- approximately 1.5% fraud rate

Therefore, this experiment intentionally reproduces the population and
label distribution of the team lead's dataset.

---
### Imports

In [1]:
import pandas as pd
from pathlib import Path

### Paths

In [2]:
PROJECT_ROOT = Path.cwd().parent.parent
DATA_PATH = PROJECT_ROOT / "data"

FULL_PATH = PROJECT_ROOT / "notebooks" / "troubleshooting" / "df_11_all_CM_fraud.csv"
TRAIN_PATH = PROJECT_ROOT / "notebooks" / "troubleshooting" / "df_11_1to12_CM_fraud.csv"
TEST_PATH = PROJECT_ROOT / "notebooks" / "troubleshooting" / "df_11_13to15_CM_fraud.csv"

### Load datasets

In [3]:
full_df = pd.read_csv(FULL_PATH)
train_df = pd.read_csv(TRAIN_PATH)

### Create test set

In [6]:
# Remove rows from full_df that have an identical row in train_df
test_df = (
    full_df
    .merge(
        train_df.drop_duplicates(),
        how="left",
        indicator=True
    )
)

test_df = test_df[
    test_df["_merge"] == "left_only"
].drop(
    columns="_merge"
)

### Save test dataset

In [8]:
test_df.to_csv(
    TEST_PATH,
    index=False
)

print(f"Test dataset saved to: {TEST_PATH}")
print(f"Test shape: {test_df.shape}")

Test dataset saved to: c:\Users\ali.farhat\Documents\simbox-project\notebooks\troubleshooting\df_11_13to15_CM_fraud.csv
Test shape: (28814, 25)
